In [1]:
import warnings
import random
import re
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, hstack
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")
np.random.seed(42)
random.seed(42)

MODEL_DIR = Path("../models")
MODEL_DIR.mkdir(exist_ok=True)
print(f"Model output directory: {MODEL_DIR.resolve()}")

# 1) Synthetic dataset generation
PHISHING_TEMPLATES = [
    {
        "subject": "URGENT: Your SBI account will be suspended",
        "body": "Dear Customer, Your SBI account has been flagged for suspicious activity. Verify immediately by clicking http://sbi-secure-verify.xyz/login. Share your User ID, Password and OTP.",
        "sender": "security@sbi-alerts.xyz",
        "urls": ["http://sbi-secure-verify.xyz/login"],
    },
    {
        "subject": "KYC Update Required - HDFC Bank",
        "body": "Your HDFC Bank KYC is expiring. Update now at http://hdfc-kyc-update.com/verify or your account will be frozen.",
        "sender": "kyc@hdfc-notifications.com",
        "urls": ["http://hdfc-kyc-update.com/verify"],
    },
    {
        "subject": "You have won Rs. 50,00,000 in RBI Lottery!",
        "body": "Claim your prize by depositing a processing fee immediately.",
        "sender": "lottery@rbi-winners.org",
        "urls": ["http://rbi-winners.org/claim"],
    },
    {
        "subject": "Your UPI account has been compromised",
        "body": "Secure your account now by verifying your UPI PIN.",
        "sender": "security@upi-secure.in",
        "urls": ["http://upi-secure.in/verify"],
    },
    {
        "subject": "Income Tax Refund - Action Required",
        "body": "You are eligible for tax refund. Enter bank details and PAN for direct deposit.",
        "sender": "refund@incometax-gov.xyz",
        "urls": ["http://incometax-refund.xyz/claim"],
    },
]

LEGIT_TEMPLATES = [
    {
        "subject": "Your January 2024 Account Statement is Ready",
        "body": "Your account statement is now available in your official netbanking portal.",
        "sender": "statements@hdfcbank.com",
        "urls": ["https://www.hdfcbank.com"],
    },
    {
        "subject": "Meeting reminder: Project review at 3 PM",
        "body": "Reminder about today's project review meeting.",
        "sender": "priya.sharma@company.com",
        "urls": [],
    },
    {
        "subject": "Your Flipkart order has been shipped",
        "body": "Track your order status at the official website.",
        "sender": "noreply@flipkart.com",
        "urls": ["https://www.flipkart.com/track"],
    },
    {
        "subject": "Your electricity bill for December 2023",
        "body": "Your MSEDCL bill is ready. Pay through the official website.",
        "sender": "billing@mahadiscom.in",
        "urls": ["https://www.mahadiscom.in"],
    },
    {
        "subject": "GitHub: New pull request in your repository",
        "body": "A new pull request has been opened. Please review changes.",
        "sender": "notifications@github.com",
        "urls": ["https://github.com/org/cybershield-ai/pull/42"],
    },
]


def augment(templates, label, n=300):
    rows = []
    urgency_words = ["URGENT", "IMMEDIATELY", "ACTION REQUIRED", "WARNING", "ALERT", "CRITICAL"]
    for _ in range(n):
        t = random.choice(templates).copy()
        text = f"{t['subject']} {t['body']}"
        if label == 1 and random.random() < 0.3:
            text = random.choice(urgency_words) + ": " + text
        if random.random() < 0.2:
            text = text.upper()
        if random.random() < 0.15:
            text = text + " Click now! Limited time offer!"
        rows.append(
            {
                "text": text,
                "sender": t.get("sender", ""),
                "urls": "|".join(t.get("urls", [])),
                "label": label,
            }
        )
    return rows


phishing_rows = augment(PHISHING_TEMPLATES, label=1, n=300)
legit_rows = augment(LEGIT_TEMPLATES, label=0, n=300)

df = pd.DataFrame(phishing_rows + legit_rows).sample(frac=1, random_state=42).reset_index(drop=True)
print(f"Dataset shape: {df.shape}")
print(f"Label distribution:\n{df['label'].value_counts()}")

# 2) Feature engineering

def extract_url_features(row):
    urls = str(row.get("urls", ""))
    sender = str(row.get("sender", ""))
    feats = {}
    feats["has_url"] = 1 if urls else 0
    feats["url_count"] = len(urls.split("|")) if urls else 0
    feats["has_http"] = 1 if "http://" in urls else 0
    feats["has_https"] = 1 if "https://" in urls else 0
    feats["has_ip"] = 1 if re.search(r"\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}", urls) else 0
    feats["suspicious_tld"] = 1 if any(urls.endswith(t) for t in [".xyz", ".tk", ".ml", ".ga", ".cf", ".top"]) else 0
    feats["url_length"] = max((len(u) for u in urls.split("|")), default=0)
    feats["sender_domain_mismatch"] = 1 if sender and not any(d in sender for d in ["gmail", "yahoo", "outlook", "hotmail", ".com", ".co.in", ".org", ".in"]) else 0
    kw = ["login", "verify", "secure", "bank", "account", "update", "kyc", "otp", "password", "confirm"]
    feats["keyword_count"] = sum(1 for k in kw if k in (urls.lower() + " " + sender.lower()))
    return feats


url_feats_df = pd.DataFrame([extract_url_features(row) for _, row in df.iterrows()])

tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    stop_words="english",
    min_df=2,
    max_df=0.95,
    sublinear_tf=True,
)

X_text = tfidf.fit_transform(df["text"])
X_url = url_feats_df.values
X_combined = hstack([X_text, csr_matrix(X_url)])
y = df["label"].values

X_train, X_test, y_train, y_test = train_test_split(
    X_combined, y, test_size=0.2, random_state=42, stratify=y
)

# 3) Train model
lr = LogisticRegression(max_iter=1000, C=1.0, random_state=42)
rf = RandomForestClassifier(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)
ensemble = VotingClassifier(
    estimators=[("lr", lr), ("rf", rf)],
    voting="soft",
    weights=[0.4, 0.6],
)

print("Training ensemble classifier...")
ensemble.fit(X_train, y_train)
print("Training complete!")

# 4) Evaluate
y_pred = ensemble.predict(X_test)
y_prob = ensemble.predict_proba(X_test)[:, 1]

print("=== Classification Report ===")
print(classification_report(y_test, y_pred, target_names=["Legitimate", "Phishing"]))
print(f"\nAUC-ROC: {roc_auc_score(y_test, y_prob):.4f}")

cm = confusion_matrix(y_test, y_pred)
print("\n=== Confusion Matrix ===")
print(f"TN={cm[0,0]} FP={cm[0,1]}")
print(f"FN={cm[1,0]} TP={cm[1,1]}")

# 5) Export artifacts
vec_path = MODEL_DIR / "phishing_tfidf_vectorizer.pkl"
clf_path = MODEL_DIR / "phishing_classifier.pkl"
joblib.dump(tfidf, vec_path)
joblib.dump(ensemble, clf_path)

print(f"Vectorizer saved: {vec_path}")
print(f"Classifier saved: {clf_path}")
print("Phishing model training complete.")

Model output directory: E:\cybershield-ai\models
Dataset shape: (600, 4)
Label distribution:
label
1    300
0    300
Name: count, dtype: int64
Training ensemble classifier...
Training complete!
=== Classification Report ===
              precision    recall  f1-score   support

  Legitimate       1.00      1.00      1.00        60
    Phishing       1.00      1.00      1.00        60

    accuracy                           1.00       120
   macro avg       1.00      1.00      1.00       120
weighted avg       1.00      1.00      1.00       120


AUC-ROC: 1.0000

=== Confusion Matrix ===
TN=60 FP=0
FN=0 TP=60
Vectorizer saved: ..\models\phishing_tfidf_vectorizer.pkl
Classifier saved: ..\models\phishing_classifier.pkl
Phishing model training complete.
